# KPConv Segmentation — Elecs Dataset Training
**Works on: Google Colab (free T4 GPU) | Kaggle (free 30h/week GPU)**

## Before you start
- **Colab**: Runtime → Change runtime type → GPU (T4)
- **Kaggle**: Settings → Accelerator → GPU T4 x2 or P100
- Upload your Elecs dataset (PLY files) to Google Drive or Kaggle Dataset, then update `DATA_DIR` below.

In [ ]:
# ── 0. Platform detection ────────────────────────────────────────────────────
import os

ON_COLAB  = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ
ON_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

print(f'Platform: {"Colab" if ON_COLAB else "Kaggle" if ON_KAGGLE else "Other"}')

import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout[:500] if result.returncode == 0 else 'No GPU detected — check runtime settings!')

In [ ]:
# ── 1. Mount Google Drive (Colab only) ──────────────────────────────────────
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # Update this to where your Elecs PLY files live on Drive:
    DATA_DIR = '/content/drive/MyDrive/Elecs/data'
else:
    # Kaggle: add your dataset from the right panel → Input → Add Dataset
    DATA_DIR = '/kaggle/input/elecs-dataset/data'

print(f'Data directory: {DATA_DIR}')
print('Files:', os.listdir(DATA_DIR) if os.path.exists(DATA_DIR) else 'NOT FOUND — update DATA_DIR above')

In [ ]:
# ── 2. Clone repo ────────────────────────────────────────────────────────────
# Replace with your actual GitHub repo URL after you push KPConv_Segmentation
REPO_URL = 'https://github.com/YOUR_USERNAME/KPConv_Segmentation.git'
REPO_DIR = '/content/KPConv_Segmentation' if ON_COLAB else '/kaggle/working/KPConv_Segmentation'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print('Repo already exists, pulling latest...')
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
# ── 3. Install Python dependencies ──────────────────────────────────────────
# Get CUDA version for correct PyTorch wheel
import subprocess
cuda_out = subprocess.run(['nvcc', '--version'], capture_output=True, text=True).stdout
print(cuda_out)

# Install PyTorch (cu121 works on Colab T4; cu128 if newer CUDA is available)
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# Install remaining dependencies
!pip install -q \
    lightning>=2.0.0 \
    torchmetrics>=0.11.4 \
    hydra-core==1.3.2 \
    hydra-colorlog==1.2.0 \
    hydra-optuna-sweeper==1.2.0 \
    rootutils \
    rich \
    scikit-learn \
    matplotlib \
    tensorboard \
    plyfile>=0.7.4 \
    numpy>=1.21.0 \
    tqdm>=4.64.0 \
    pybind11>=2.10.0 \
    setuptools>=60.0.0

print('Dependencies installed.')

In [ ]:
# ── 4. Compile C++ extensions (required!) ────────────────────────────────────
import subprocess, sys

def build_ext(ext_dir):
    result = subprocess.run(
        [sys.executable, 'setup.py', 'build_ext', '--inplace'],
        cwd=ext_dir, capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f'ERROR building {ext_dir}:')
        print(result.stderr[-2000:])
    else:
        print(f'OK: {ext_dir}')

base = os.path.join(REPO_DIR, 'src', 'data', 'cpp_wrappers')
build_ext(os.path.join(base, 'cpp_subsampling'))
build_ext(os.path.join(base, 'cpp_neighbors'))

In [ ]:
# ── 5. Install the package in editable mode ──────────────────────────────────
!pip install -q -e .
print('Package installed.')

In [ ]:
# ── 6. Update data path in config ────────────────────────────────────────────
import re

paths_cfg = os.path.join(REPO_DIR, 'configs', 'paths', 'default.yaml')
with open(paths_cfg) as f:
    content = f.read()

# Replace the hard-coded Windows path with the cloud data path
content = re.sub(
    r's3dis_elecs_data_dir:.*',
    f's3dis_elecs_data_dir: "{DATA_DIR}"',
    content
)

with open(paths_cfg, 'w') as f:
    f.write(content)

print('Updated paths/default.yaml:')
print(content)

In [ ]:
# ── 7. Quick smoke test (fast_dev_run — no real training, just checks pipeline) ─
# Runs 1 train batch + 1 val batch to confirm everything works before full training
!python src/train_elecs.py trainer.fast_dev_run=True trainer.accelerator=gpu

In [ ]:
# ── 8. Full training ─────────────────────────────────────────────────────────
# Adjust max_epochs and batch_size as needed.
# On Colab T4: batch_size=4-6 is safe. On Kaggle P100: batch_size=8.
# Logs go to logs/ directory. Open TensorBoard below to monitor.

!python src/train_elecs.py \
    trainer.accelerator=gpu \
    trainer.devices=1 \
    trainer.max_epochs=20 \
    common.batch_size=6 \
    common.num_workers=2

In [ ]:
# ── 9. TensorBoard monitoring ────────────────────────────────────────────────
%load_ext tensorboard
%tensorboard --logdir logs/

In [ ]:
# ── 10. Save checkpoint to Drive (Colab) or output (Kaggle) ──────────────────
import shutil, glob

checkpoints = glob.glob(os.path.join(REPO_DIR, 'logs', '**', '*.ckpt'), recursive=True)
print('Checkpoints found:', checkpoints)

if ON_COLAB and checkpoints:
    save_dir = '/content/drive/MyDrive/KPConv_checkpoints'
    os.makedirs(save_dir, exist_ok=True)
    for ckpt in checkpoints:
        shutil.copy2(ckpt, save_dir)
    print(f'Saved {len(checkpoints)} checkpoint(s) to Drive: {save_dir}')
elif ON_KAGGLE and checkpoints:
    # Kaggle auto-saves /kaggle/working/ when the notebook finishes
    print('Kaggle: checkpoints will be saved at end of session in /kaggle/working/')
    for ckpt in checkpoints:
        print(' -', ckpt)